# W1D1: 이미지 로딩 · 저장 · 표시

## 📌 오늘의 목표
- NEU Steel Surface Defect 6종 결함 이미지를 로딩하고 기본 정보 확인
- 해상도(height x width), 채널 수, dtype 등 이미지 속성 파악
- matplotlib으로 2x3 그리드에 6종 결함을 한눈에 비교

## 🎯 배울 함수
| 함수 | 설명 |
|------|------|
| `cv2.imread(path, flag)` | 이미지 파일을 numpy 배열로 로딩 |
| `cv2.imwrite(path, img)` | numpy 배열을 이미지 파일로 저장 |
| `.shape` | (height, width, channels) 튜플 반환 |
| `plt.subplots(nrows, ncols)` | 여러 이미지를 그리드로 배치 |

## 📦 import + 데이터 경로 설정

필요한 라이브러리를 불러오고, NEU 데이터셋 경로를 설정합니다.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

DATA_DIR = Path("../data/kaggle/archive/NEU-DET/train/images")
DEFECT_TYPES = ["crazing", "inclusion", "patches", "pitted_surface", "rolled-in_scale", "scratches"]

## 🔧 1. 이미지 한 장 로딩해보기

**`cv2.imread(path, flag)`** 로 이미지를 읽습니다.
- `cv2.IMREAD_COLOR` (기본값): BGR 3채널로 로딩
- `cv2.IMREAD_GRAYSCALE`: 흑백 1채널로 로딩
- 반환값은 numpy 배열 (없으면 None)

**제조 검사 연결:** AOI 장비도 카메라에서 이미지를 받아 numpy 배열로 변환하는 것이 첫 단계입니다.

In [ ]:
img_path = DATA_DIR / "crazing" / "crazing_1.jpg"
img = cv2.imread(str(img_path))

print(f"파일 경로: {img_path}")
print(f"타입: {type(img)}")
print(f"shape: {img.shape}")
print(f"dtype: {img.dtype}")
print(f"해상도: {img.shape[1]}(W) x {img.shape[0]}(H)")
print(f"채널 수: {img.shape[2]}")
print(f"픽셀값 범위: {img.min()} ~ {img.max()}")

### 해석 가이드
- **shape = (H, W, C)**: OpenCV는 항상 (높이, 너비, 채널) 순서. 행렬 관점이라 (row, col)
- **dtype = uint8**: 픽셀값 0~255 범위의 8비트 정수
- **BGR 순서**: OpenCV는 RGB가 아니라 BGR! matplotlib으로 표시할 때 변환 필요

> 🧪 **실험해보기:** `cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)`로 바꿔서 shape이 어떻게 달라지는지 확인해보세요.

In [ ]:
img_color = cv2.imread(str(img_path))
img_gray = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)

print(f"컬러 shape: {img_color.shape}  → {img_color.shape[0]*img_color.shape[1]*img_color.shape[2]:,} 픽셀값")
print(f"흑백 shape:  {img_gray.shape}     → {img_gray.shape[0]*img_gray.shape[1]:,} 픽셀값")
print(f"메모리: 컬러 {img_color.nbytes:,} bytes vs 흑백 {img_gray.nbytes:,} bytes (1/3)")

## 🔧 2. 6종 결함 이미지 로딩 + 정보 출력

6종 결함에서 각 1장씩 로딩하고, 이미지 정보를 한눈에 비교합니다.

**제조 검사 연결:** 실제 AOI에서도 결함 유형별로 이미지 특성(밝기, 텍스처)이 다릅니다. 이 차이를 눈으로 먼저 파악하는 것이 중요합니다.

In [ ]:
images = {}

for defect in DEFECT_TYPES:
    path = DATA_DIR / defect / f"{defect}_1.jpg"
    img = cv2.imread(str(path))
    images[defect] = img
    print(f"{defect:20s} | shape: {img.shape} | dtype: {img.dtype} | 밝기 평균: {img.mean():.1f}")

## 🔧 3. 2x3 그리드로 6종 결함 시각화

**`plt.subplots(nrows, ncols)`** 로 여러 이미지를 한 화면에 배치합니다.
- OpenCV는 BGR이므로 `cv2.cvtColor(img, cv2.COLOR_BGR2RGB)`로 변환 후 표시
- `ax.set_title()`로 결함 유형 표시
- `ax.axis('off')`로 축 숨김

**제조 검사 연결:** 검사 알고리즘 개발 시 결함 유형별 이미지를 나란히 놓고 비교하는 것은 기본 중의 기본입니다.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

for ax, (defect, img) in zip(axes.flat, images.items()):
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    ax.imshow(img_rgb)
    ax.set_title(defect, fontsize=14, fontweight='bold')
    ax.axis('off')

plt.suptitle("NEU Steel Surface Defect - 6 Types", fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig("W1D1_6types_grid.png", dpi=150, bbox_inches='tight')
plt.show()

### 해석 가이드
- **crazing**: 거미줄처럼 미세한 균열 패턴. 배경과 대비가 낮아 검출 어려움
- **inclusion**: 이물질이 표면에 포함된 형태. 비교적 뚜렷한 밝기 차이
- **patches**: 얼룩/반점 형태. 넓은 영역에 걸쳐 나타남
- **pitted_surface**: 표면 움푹 파인 결함. 작은 점들이 산재
- **rolled-in_scale**: 압연 스케일이 말려 들어간 형태. 길쭉한 패턴
- **scratches**: 긁힌 자국. 선형 패턴이 뚜렷

> 🤔 **판단 질문:** 6종 중 육안으로 가장 구분하기 쉬운 결함은? 가장 어려운 결함은? 그 이유는?

## 🔧 4. 이미지 저장 (imwrite)

**`cv2.imwrite(path, img)`** 로 numpy 배열을 이미지 파일로 저장합니다.
- BGR 순서 그대로 저장 (OpenCV끼리는 문제 없음)
- 저장 포맷은 확장자로 자동 결정 (.jpg, .png 등)
- 반환값: 성공 시 True

**제조 검사 연결:** 검사 결과 이미지를 저장해두면 나중에 불량 원인 분석(사후 추적)에 활용합니다.

In [ ]:
save_path = "W1D1_crazing_copy.png"
result = cv2.imwrite(save_path, images["crazing"])
print(f"저장 결과: {result}")
print(f"저장 경로: {save_path}")

img_reload = cv2.imread(save_path)
print(f"다시 로딩한 shape: {img_reload.shape}")
print(f"원본과 동일한가: {np.array_equal(images['crazing'], img_reload)}")

### 해석 가이드
- 원본은 `.jpg`(손실 압축), 저장은 `.png`(무손실)로 했기 때문에 픽셀값이 동일할 수도 있고 아닐 수도 있음
- jpg→png 변환 시에는 동일하지만, png→jpg→png는 손실 발생

> 🧪 **실험해보기:** `.jpg`로 저장해서 다시 로딩하면 원본과 동일한가? `quality` 파라미터를 바꿔보세요:
> `cv2.imwrite("test.jpg", img, [cv2.IMWRITE_JPEG_QUALITY, 50])`

## 🔬 파라미터 실험: 같은 결함, 다른 이미지

같은 결함 유형이라도 이미지마다 모양이 다릅니다. 결함 1장이 아닌 여러 장을 비교해봅시다.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
sample_ids = [1, 5, 10, 50, 100, 200]

for ax, sid in zip(axes.flat, sample_ids):
    path = DATA_DIR / "scratches" / f"scratches_{sid}.jpg"
    img = cv2.imread(str(path))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    ax.imshow(img_rgb)
    ax.set_title(f"scratches_{sid}", fontsize=12)
    ax.axis('off')

plt.suptitle("같은 결함(scratches), 다른 이미지", fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

### 해석 가이드
- 같은 "scratches"라도 긁힘 방향, 깊이, 개수가 모두 다름
- 이것이 제조 검사의 핵심 난이도: **같은 결함 유형이라도 변동성(variation)이 크다**
- 단순 규칙 하나로 모든 케이스를 잡기 어려운 이유

> 🧪 **실험해보기:** `"scratches"`를 다른 결함 유형으로 바꿔보세요. 어떤 결함이 이미지마다 가장 많이 다른가요?

## 📝 정리

### 오늘 배운 것
| 함수 | 핵심 포인트 |
|------|------------|
| `cv2.imread()` | BGR 순서로 로딩, None 체크 필수 |
| `cv2.imwrite()` | 확장자로 포맷 결정, jpg는 손실 압축 |
| `.shape` | (H, W, C) 순서 — 높이가 먼저! |
| `plt.subplots()` | 여러 이미지 비교에 필수 |
| `cv2.cvtColor(BGR2RGB)` | matplotlib 표시 시 반드시 변환 |

### 핵심 개념
- OpenCV 이미지 = numpy 배열 (H, W, C), dtype=uint8, 값 범위 0~255
- BGR 순서 주의 (matplotlib은 RGB)
- NEU 데이터셋: 6종 결함, 각 240장, 같은 유형 내에서도 변동성 큼

### 복습 퀴즈
1. `cv2.imread()`로 읽은 이미지의 채널 순서는 ___이다. (RGB / BGR)
2. `img.shape`이 `(200, 300, 3)`이면 높이는 ___, 너비는 ___ 이다.
3. matplotlib에서 OpenCV 이미지를 올바르게 표시하려면 `cv2.cvtColor(img, cv2.COLOR_____)`를 사용해야 한다.

### 다음에 할 것 (W1D2)
- 색공간 변환 (BGR → Grayscale, HSV)
- ROI(Region of Interest) 슬라이싱으로 결함 부분만 잘라서 확대